# 50D skew-symmetric product-form benchmark

This notebook is the compact, self-contained reference implementation of the
50-dimensional Holomorphic Cauchy Network (HCN) experiment. It reconstructs the
scientific path used by the completed 100k run while removing local Windows
paths, best-checkpoint files, retrospective 50k-to-86k checkpoint diagnostics,
and saved training output.

The paper endpoint is fixed **before training** at optimizer step 100,000. The
50D batches are half of the 30D batches and the number of updates is doubled,
so both runs expose every loss branch to the same 64M sampled constraints.
Ground truth first appears after training. Talbot Degree 8 is the primary
reported inversion and Degree 5 is retained as a numerical diagnostic.

The safe default is a two-step smoke test. A formal run requires explicitly
setting `RUN_MODE = "train"`; `RUN_MODE = "evaluate"` loads one fixed endpoint
without training or modifying it. The archived August 2026 checkpoint uses a
legacy payload and is accepted only after its released SHA-256 is placed in the
frozen configuration below.


## 1. Imports and reproducibility


In [ ]:
import os
os.environ.setdefault("KMP_DUPLICATE_LIB_OK", "TRUE")
os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")

import csv
import hashlib
import json
import math
import platform
import random
import sys
import time
from dataclasses import dataclass
from datetime import datetime, timezone
from pathlib import Path

import matplotlib.pyplot as plt
import mpmath as mp
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
r_dtype = torch.float64
c_dtype = torch.complex128
torch.set_default_dtype(r_dtype)
torch.use_deterministic_algorithms(True, warn_only=True)
if torch.cuda.is_available():
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True


def set_all_seeds(seed):
    seed = int(seed)
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def utc_stamp():
    return datetime.now(timezone.utc).strftime("%Y%m%d-%H%M%S")


def write_json(path, payload):
    Path(path).write_text(
        json.dumps(payload, indent=2, allow_nan=False), encoding="utf-8"
    )


def write_csv(path, rows, fieldnames=None):
    rows = list(rows)
    if not rows and fieldnames is None:
        raise ValueError("fieldnames are required when rows is empty")
    names = list(fieldnames or rows[0].keys())
    with Path(path).open("w", newline="", encoding="utf-8") as handle:
        writer = csv.DictWriter(handle, fieldnames=names)
        writer.writeheader()
        writer.writerows(rows)


def sha256_file(path, chunk_size=1024 * 1024):
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        while True:
            chunk = handle.read(chunk_size)
            if not chunk:
                break
            digest.update(chunk)
    return digest.hexdigest()


print({
    "device": str(device),
    "torch": torch.__version__,
    "complex_dtype": str(c_dtype),
})


## 2. Frozen paper configuration


In [ ]:
# Safe public default. Use "train" only for a formal 100k run.
RUN_MODE = "smoke"                 # "smoke", "train", "evaluate"
EVALUATE_RUN_DIR = None            # required for RUN_MODE == "evaluate"
RESUME_CHECKPOINT = None           # optional .../seed_12345/resume.pt for "train"

ALLOWED_RUN_MODES = {"smoke", "train", "evaluate"}
if RUN_MODE not in ALLOWED_RUN_MODES:
    raise ValueError(f"RUN_MODE must be one of {sorted(ALLOWED_RUN_MODES)}")
if RUN_MODE == "evaluate" and EVALUATE_RUN_DIR is None:
    raise ValueError("Set EVALUATE_RUN_DIR before RUN_MODE='evaluate'.")
if RUN_MODE != "train" and RESUME_CHECKPOINT is not None:
    raise ValueError("RESUME_CHECKPOINT is used only with RUN_MODE='train'.")

EXPERIMENT_SLUG = "d50_skew_symmetric_product_form"
OUTPUT_ROOT = Path("outputs")
DIMENSION = 50
PAPER_SEED = 12345
TRAIN_STEPS = 100_000
LR_SWITCH_AFTER_STEP = 80_000
CHECKPOINT_STEP = 100_000
LOG_EVERY = 1_000
RECOVERY_EVERY = 1_000
FACE_BLOCK_SIZE = 4
HISTORICAL_CHECKPOINT_SCHEMA = "d50_general_pointmatched100k_v1"
HISTORICAL_EXPERIMENT_VARIANT = "d50_controlled_general_scaling_pure_general"
# Exact archived endpoint used by the independent fixed-checkpoint diagnostic.
HISTORICAL_CHECKPOINT_SHA256 = "669c9bc1b2188241e47e78ee5a23bfaa690a86233529b8eef340daf11cf79b0a"

MODEL_CONFIG = {
    "d": DIMENSION,
    "coord_embed_dim": 16,
    "boundary_embed_dim": 16,
    "interior_hidden_width": 128,
    "boundary_hidden_width": 128,
    "pair_chunk_size": 32_768,
    "face_block_size": FACE_BLOCK_SIZE,
    "boundary_anchor_mode": "hard_rbm_flow_balance",
}

# Flat keys preserve the scientific names used by the original implementation.
TRAIN_CONFIG = {
    "d": DIMENSION,
    "seed": PAPER_SEED,
    "eval_seed": PAPER_SEED + 1_000_003,
    "coord_embed_dim": MODEL_CONFIG["coord_embed_dim"],
    "boundary_embed_dim": MODEL_CONFIG["boundary_embed_dim"],
    "interior_hidden_width": MODEL_CONFIG["interior_hidden_width"],
    "boundary_hidden_width": MODEL_CONFIG["boundary_hidden_width"],
    "pair_chunk_size": MODEL_CONFIG["pair_chunk_size"],
    "batch_bar": 512,
    "batch_bdary": 64,
    "batch_mono": 64,
    "eval_batch_bar": 1_024,
    "eval_batch_bdary": 128,
    "eval_batch_mono": 128,
    "lambda_bar": 1.0,
    "lambda_bdary": 1.0,
    "lambda_zero": 0.0,
    "lambda_mono": 1.0,
    "lambda_imag": 1.0,
    "mono_re_min": -0.5,
    "mono_re_max": 5.0,
    "adaptive_bar_weight": True,
    "bar_weight_alpha": 0.3,
    "bar_weight_eps": 1.0e-12,
    "bar_weight_min": 0.5,
    "bar_weight_max": 2.0,
    "lr_stage1": 5.0e-5,
    "lr_stage2": 1.0e-5,
    "lr_switch_after_step": LR_SWITCH_AFTER_STEP,
    "grad_clip": 1.0,
}

SAMPLER_CONFIG = {
    "name": "original_general_mixed_BAR",
    "real_domain": [-0.5, 5.0],
    "imaginary_abs_max": 5.0,
    "bar_mixture": {
        "general_complex": 0.50,
        "exact_complex_diagonal": 0.20,
        "near_complex_diagonal": 0.20,
        "positive_real_diagonal": 0.10,
    },
    "near_diagonal_noise_scale": 0.05,
    "contains_special_inversion_nodes": False,
}

EVALUATION_CONFIG = {
    "t_values": list(range(1, 11)),
    "talbot_degrees": [5, 8],
    "primary_degree": 8,
    "degree5_role": "inversion diagnostic",
    "raw_predictions": True,
    "prediction_clipping": False,
    "mpmath_decimal_digits": 100,
}

PER_RUN_SAMPLE_BUDGET = TRAIN_STEPS * (
    TRAIN_CONFIG["batch_bar"]
    + TRAIN_CONFIG["batch_bdary"]
    + TRAIN_CONFIG["batch_mono"]
)
SAMPLE_BUDGET = {
    "bar_points": TRAIN_STEPS * TRAIN_CONFIG["batch_bar"],
    "boundary_points": TRAIN_STEPS * TRAIN_CONFIG["batch_bdary"],
    "monotonicity_points": TRAIN_STEPS * TRAIN_CONFIG["batch_mono"],
    "total_sampled_constraints": PER_RUN_SAMPLE_BUDGET,
    "matched_30d_total_sampled_constraints": 64_000_000,
}
assert PER_RUN_SAMPLE_BUDGET == 64_000_000

RUN_CONFIG = {
    "experiment_slug": EXPERIMENT_SLUG,
    "dimension": DIMENSION,
    "paper_seed": PAPER_SEED,
    "model": MODEL_CONFIG,
    "training": TRAIN_CONFIG,
    "sampler": SAMPLER_CONFIG,
    "evaluation": EVALUATION_CONFIG,
    "checkpoint_selection": "prespecified fixed step 100000 only",
    "recovery_checkpoints_used_for_selection": False,
    "ground_truth_used_by_training": False,
    "ground_truth_used_for_checkpoint_selection": False,
    "audited_historical_checkpoint_schema": HISTORICAL_CHECKPOINT_SCHEMA,
    "audited_historical_checkpoint_sha256": HISTORICAL_CHECKPOINT_SHA256,
}
print({
    "run_mode": RUN_MODE,
    "seed": PAPER_SEED,
    "fixed_endpoint": CHECKPOINT_STEP,
    "sample_budget": PER_RUN_SAMPLE_BUDGET,
})


## 3. RBM benchmark and mixed complex sampler


In [ ]:
# RBM parameters, gamma convention, general samplers, and direction cache

@dataclass
class RBMParams:
    mu: torch.Tensor
    Sigma: torch.Tensor
    R: torch.Tensor
    alpha: torch.Tensor
    delta: torch.Tensor | None = None
    a: torch.Tensor | None = None

def make_skew_symmetric_product_form_params(d, device=device, r_dtype=r_dtype, c_dtype=c_dtype):
    R = torch.zeros(d, d, dtype=r_dtype, device=device)
    Sigma = torch.zeros(d, d, dtype=r_dtype, device=device)
    idx = torch.arange(d, device=device)
    R[idx, idx] = 1.0
    Sigma[idx, idx] = 2.0
    for i in range(1, d):
        R[i, i - 1] = -1.0
        Sigma[i, i - 1] = -1.0
        Sigma[i - 1, i] = -1.0

    mu = -torch.ones(d, dtype=r_dtype, device=device)
    alpha = torch.arange(1, d + 1, dtype=r_dtype, device=device)
    alpha_formula = -2.0 * torch.diag(R) * torch.linalg.solve(R, mu) / torch.diag(Sigma)
    if not torch.allclose(alpha_formula, alpha, rtol=1e-12, atol=1e-12):
        raise AssertionError("Formula alpha check failed.")
    delta = torch.diag(Sigma) / (2.0 * torch.diag(R)) * alpha
    return RBMParams(mu=mu, Sigma=Sigma, R=R, alpha=alpha, delta=delta, a=alpha)

def compute_boundary_mass(params, tol=1e-12):
    mass = -torch.linalg.solve(
        params.R.to(dtype=r_dtype, device=device),
        params.mu.to(dtype=r_dtype, device=device),
    )
    if mass.ndim != 1:
        raise ValueError(f"Boundary mass must be one-dimensional, got shape {tuple(mass.shape)}")
    if not torch.isfinite(mass).all():
        raise ValueError("Boundary mass contains non-finite entries.")
    if torch.any(mass <= tol):
        raise ValueError(
            "Boundary rates must be strictly positive. "
            "Check RBM stability and the R/mu sign convention."
        )
    return mass

def compute_gammas(theta, params):
    theta = theta.to(dtype=c_dtype, device=device)
    mu_c = params.mu.to(dtype=c_dtype, device=device)
    Sigma_c = params.Sigma.to(dtype=c_dtype, device=device)
    R_c = params.R.to(dtype=c_dtype, device=device)
    gamma0 = -0.5 * torch.sum((theta @ Sigma_c) * theta, dim=1, keepdim=True) + theta @ mu_c.view(-1, 1)
    gamma_all = -(theta @ R_c)  # column-based convention
    return gamma0, gamma_all

def sample_complex_box(N, d=50, re_min=-0.5, re_max=5.0, im_max=5.0, requires_grad=False, device=device):
    real = re_min + (re_max - re_min) * torch.rand(N, d, dtype=r_dtype, device=device)
    imag = -im_max + 2.0 * im_max * torch.rand(N, d, dtype=r_dtype, device=device)
    theta = torch.complex(real, imag)
    if requires_grad:
        theta.requires_grad_(True)
    return theta

def sample_nn4mgf_vector_mgf(N, d=50, lb=-5.0, ub=0.5, imag_lb=-5.0, imag_ub=5.0, device=device):
    real_lb_curr = ub - (ub - lb) * torch.rand(N, 1, dtype=r_dtype, device=device)
    imag_bd = max(abs(imag_ub), abs(imag_lb)) * torch.rand(N, 1, dtype=r_dtype, device=device)
    real = (ub - real_lb_curr) * torch.rand(N, d, dtype=r_dtype, device=device) + real_lb_curr
    imag = 2.0 * imag_bd * torch.rand(N, d, dtype=r_dtype, device=device) - imag_bd
    return torch.complex(real, imag)

def sample_nn4mgf_vector_laplace(N, d=50, lap_re_min=-0.5, lap_re_max=5.0, im_max=5.0, requires_grad=False, device=device):
    theta = -sample_nn4mgf_vector_mgf(N=N, d=d, lb=-lap_re_max, ub=-lap_re_min, imag_lb=-im_max, imag_ub=im_max, device=device)
    if requires_grad:
        theta.requires_grad_(True)
    return theta

def sample_training_theta(N, d=50, re_min=-0.5, re_max=5.0, im_max=5.0, requires_grad=False, device=device):
    return sample_nn4mgf_vector_laplace(N=N, d=d, lap_re_min=re_min, lap_re_max=re_max, im_max=im_max, requires_grad=requires_grad, device=device)

def sample_nested_real_radius(N, lb=0.0, ub=5.0, device=device):
    # Near-zero-biased sampler: E[s] = lb + (ub-lb)/3 for U(0,1)^2.
    u = torch.rand(N, 1, dtype=r_dtype, device=device)
    return lb + (ub - lb) * (u ** 2)

def sample_mixed_bar_theta(N=4000, d=50, re_min=-0.5, re_max=5.0, im_max=5.0, device=device):
    """
    Mixed 4000-point BAR sampler proportions:
      2000 global, 800 exact complex diagonal, 800 near-diagonal complex, 400 positive-real diagonal.
    Counts are scaled proportionally for N != 4000.
    """
    n_global = int(round(N * 0.50))
    n_exact_diag = int(round(N * 0.20))
    n_near_diag = int(round(N * 0.20))
    n_pos_diag = N - n_global - n_exact_diag - n_near_diag

    global_theta = sample_training_theta(n_global, d=d, re_min=re_min, re_max=re_max, im_max=im_max, device=device)
    s_complex = sample_training_theta(n_exact_diag, d=1, re_min=re_min, re_max=re_max, im_max=im_max, device=device)
    exact_diag = s_complex.expand(n_exact_diag, d)

    s_near = sample_training_theta(n_near_diag, d=1, re_min=re_min, re_max=re_max, im_max=im_max, device=device)
    noise = 0.05 * sample_complex_box(n_near_diag, d=d, re_min=-1.0, re_max=1.0, im_max=1.0, device=device)
    near_diag_raw = s_near.expand(n_near_diag, d) + noise
    near_diag = torch.complex(
        torch.clamp(near_diag_raw.real, min=re_min, max=re_max),
        torch.clamp(near_diag_raw.imag, min=-im_max, max=im_max),
    )

    positive_lb = max(0.0, re_min)
    positive_ub = re_max
    if positive_ub <= positive_lb:
        raise ValueError(f"Positive-real diagonal domain is empty: [{positive_lb}, {positive_ub}]")
    s_real = sample_nested_real_radius(n_pos_diag, lb=positive_lb, ub=positive_ub, device=device)
    pos_diag = torch.complex(s_real.expand(n_pos_diag, d), torch.zeros(n_pos_diag, d, dtype=r_dtype, device=device))

    theta = torch.cat([global_theta, exact_diag, near_diag, pos_diag], dim=0).to(dtype=c_dtype, device=device)
    return theta[torch.randperm(theta.shape[0], device=device)]

def construct_boundary_directions(
    params,
    target_device=None,
    target_dtype=None,
):
    target_device = torch.device(
        device if target_device is None else target_device
    )
    target_dtype = c_dtype if target_dtype is None else target_dtype
    R_c = params.R.to(
        dtype=target_dtype,
        device=target_device,
    )
    matrix = R_c.T
    d = R_c.shape[0]
    eye = torch.eye(
        d,
        dtype=target_dtype,
        device=target_device,
    )
    rows = []
    for k in range(d):
        constrained = matrix.clone()
        constrained[k, :] = eye[k, :]
        rows.append(
            torch.linalg.solve(
                constrained,
                eye[k, :],
            ).view(1, d)
        )
    return torch.cat(rows, dim=0)

def _boundary_cache_key(target_device, target_dtype):
    target_device = torch.device(target_device)
    return str(target_device), str(target_dtype)

def get_boundary_directions_cached(
    params,
    target_device=None,
    target_dtype=c_dtype,
):
    target_device = torch.device(
        device if target_device is None else target_device
    )
    target_dtype = c_dtype if target_dtype is None else target_dtype
    key = _boundary_cache_key(target_device, target_dtype)
    cache = getattr(params, "_boundary_directions_cache", None)
    if cache is None or not isinstance(cache, dict):
        cache = {}
        params._boundary_directions_cache = cache
    builds = getattr(params, "_boundary_directions_cache_builds", None)
    if builds is None or not isinstance(builds, dict):
        builds = {}
        params._boundary_directions_cache_builds = builds
    if key not in cache:
        cache[key] = construct_boundary_directions(
            params,
            target_device=target_device,
            target_dtype=target_dtype,
        ).detach()
        builds[key] = int(builds.get(key, 0)) + 1
    return cache[key]

def clear_boundary_direction_cache(params):
    params._boundary_directions_cache = {}
    params._boundary_directions_cache_builds = {}

def construct_boundary_thetas(
    theta,
    params,
    boundary_directions=None,
):
    theta = theta.to(dtype=c_dtype, device=device)
    _, d = theta.shape
    directions = (
        boundary_directions
        if boundary_directions is not None
        else get_boundary_directions_cached(
            params,
            target_device=theta.device,
            target_dtype=theta.dtype,
        )
    )
    directions = directions.to(
        device=theta.device,
        dtype=theta.dtype,
    )
    return [
        theta[:, k:k + 1] * directions[k:k + 1, :]
        for k in range(d)
    ], directions


PARAMS = make_skew_symmetric_product_form_params(DIMENSION)
BOUNDARY_MASS = compute_boundary_mass(PARAMS)
EXPECTED_BOUNDARY_MASS = torch.arange(
    1, DIMENSION + 1, dtype=r_dtype, device=device
)
if not torch.allclose(
    BOUNDARY_MASS, EXPECTED_BOUNDARY_MASS, atol=1.0e-12, rtol=0.0
):
    raise AssertionError(
        "The audited 50D boundary masses must equal rates 1,...,50."
    )

RBM_AUDIT = {
    "dimension": int(PARAMS.mu.numel()),
    "mu": PARAMS.mu.detach().cpu().tolist(),
    "sigma": PARAMS.Sigma.detach().cpu().tolist(),
    "reflection_matrix": PARAMS.R.detach().cpu().tolist(),
    "boundary_mass": BOUNDARY_MASS.detach().cpu().tolist(),
}
print({
    "dimension": int(PARAMS.mu.numel()),
    "boundary_mass": BOUNDARY_MASS.detach().cpu().tolist(),
})


## 4. Coordinate-shared Holomorphic Cauchy Network

The state-dict-compatible structured model uses face blocking and call-local
caching to control 50D memory. The interior and boundary transforms share two
scalar Cauchy networks; coordinate and boundary embeddings distinguish their
roles. Origin and boundary-mass anchors are structural rather than fitted.


In [ ]:
# Holomorphic Cauchy activation and structured-coordinate model

class CauchyComplex(nn.Module):
    def __init__(self, num_features, init_lambda=0.1, init_d=0.5, d_min=1e-3):
        super().__init__()
        self.num_features = num_features
        self.d_min = d_min
        self.lambda1 = nn.Parameter(
            torch.ones(num_features, dtype=c_dtype) * complex(init_lambda, 0.0)
        )
        self.lambda2 = nn.Parameter(
            torch.ones(num_features, dtype=c_dtype) * complex(init_lambda, 0.0)
        )
        init_value = max(init_d - d_min, 1e-6)
        init_rho = torch.log(torch.expm1(torch.tensor(init_value, dtype=r_dtype)))
        self.rho = nn.Parameter(
            torch.ones(num_features, dtype=r_dtype) * init_rho
        )
        self.last_min_abs_denominator = None
        self.running_min_abs_denominator = None

    def positive_d(self):
        return F.softplus(self.rho) + self.d_min

    def reset_denominator_tracking(self):
        self.last_min_abs_denominator = None
        self.running_min_abs_denominator = None

    def _update_denominator_tracking(self, denom):
        current = torch.min(torch.abs(denom)).detach()
        self.last_min_abs_denominator = current
        if self.running_min_abs_denominator is None:
            self.running_min_abs_denominator = current
        else:
            self.running_min_abs_denominator = torch.minimum(
                self.running_min_abs_denominator,
                current,
            ).detach()

    def _denominator(self, z):
        d_complex = self.positive_d().to(
            device=z.device,
            dtype=z.dtype,
        )
        return z ** 2 + d_complex ** 2, d_complex

    def _value_from_denominator(self, z, denom):
        return (self.lambda1 * z + self.lambda2) / denom

    def _derivative_from_denominator(self, z, denom, d_complex):
        return (
            self.lambda1 * (d_complex ** 2 - z ** 2)
            - 2.0 * self.lambda2 * z
        ) / (denom ** 2)

    @staticmethod
    def _assert_finite(value, label):
        if (
            not torch.isfinite(value.real).all()
            or not torch.isfinite(value.imag).all()
        ):
            raise FloatingPointError(f"Non-finite Cauchy {label} output.")

    def forward(self, z):
        denom, _ = self._denominator(z)
        self._update_denominator_tracking(denom)
        value = self._value_from_denominator(z, denom)
        self._assert_finite(value, "activation")
        return value

    def derivative(self, z):
        denom, d_complex = self._denominator(z)
        self._update_denominator_tracking(denom)
        derivative = self._derivative_from_denominator(
            z,
            denom,
            d_complex,
        )
        self._assert_finite(derivative, "derivative")
        return derivative

    def forward_and_derivative(self, z):
        denom, d_complex = self._denominator(z)
        self._update_denominator_tracking(denom)
        value = self._value_from_denominator(z, denom)
        derivative = self._derivative_from_denominator(
            z,
            denom,
            d_complex,
        )
        self._assert_finite(value, "activation")
        self._assert_finite(derivative, "derivative")
        return value, derivative

class CoordinateCauchyScalarNet(nn.Module):
    """
    Shared scalar complex network:
        Linear(input_dim,H) -> CauchyComplex(H) -> Linear(H,1).
    First input column is the scalar coordinate theta_i.
    """

    def __init__(self, input_dim, hidden_width):
        super().__init__()
        self.input_dim = input_dim
        self.hidden_width = hidden_width
        self.linear_0 = nn.Linear(
            input_dim,
            hidden_width,
            dtype=c_dtype,
        )
        self.cauchy_0 = CauchyComplex(hidden_width)
        self.linear_1 = nn.Linear(
            hidden_width,
            1,
            dtype=c_dtype,
        )

    def forward(self, x):
        if x.ndim != 2 or x.shape[1] != self.input_dim:
            raise ValueError(
                f"x must have shape [N,{self.input_dim}], "
                f"got {tuple(x.shape)}"
            )
        x = x.to(dtype=c_dtype, device=device)
        return self.linear_1(
            self.cauchy_0(self.linear_0(x))
        )

    def forward_with_first_input_derivative(self, x):
        x = x.to(dtype=c_dtype, device=device)
        z = self.linear_0(x)
        phi, phi_prime = self.cauchy_0.forward_and_derivative(z)
        out = self.linear_1(phi)
        deriv = torch.einsum(
            "h,nh,h->n",
            self.linear_1.weight[0],
            phi_prime,
            self.linear_0.weight[:, 0],
        ).unsqueeze(1)
        return out, deriv

def run_scalar_net_chunked(net, x, pair_chunk_size=32768, with_derivative=False):
    outs = []
    derivs = []
    for start in range(0, x.shape[0], pair_chunk_size):
        chunk = x[start:start + pair_chunk_size]
        if with_derivative:
            out, deriv = net.forward_with_first_input_derivative(chunk)
            outs.append(out)
            derivs.append(deriv)
        else:
            outs.append(net(chunk))
    if with_derivative:
        return torch.cat(outs, dim=0), torch.cat(derivs, dim=0)
    return torch.cat(outs, dim=0)

class StructuredCoordinateCauchyMGF(nn.Module):
    """
    State-dict-compatible structured-coordinate model.

    Fixed masks and indices are non-persistent buffers. They therefore do
    not add state-dict keys and do not change the architecture fingerprint.
    """

    def __init__(
        self,
        d=50,
        coord_embed_dim=16,
        boundary_embed_dim=16,
        interior_hidden_width=128,
        boundary_hidden_width=128,
        pair_chunk_size=32768,
        boundary_mass=None,
        face_block_size=FACE_BLOCK_SIZE,
    ):
        super().__init__()
        self.d = int(d)
        self.coord_embed_dim = int(coord_embed_dim)
        self.boundary_embed_dim = int(boundary_embed_dim)
        self.pair_chunk_size = int(pair_chunk_size)
        self.face_block_size = int(face_block_size)
        if self.face_block_size < 1:
            raise ValueError("face_block_size must be positive.")
        if boundary_mass is None:
            raise ValueError(
                "StructuredCoordinateCauchyMGF requires boundary_mass."
            )
        boundary_mass_real = boundary_mass.detach().clone().to(
            dtype=r_dtype,
            device=device,
        )
        if boundary_mass_real.shape != (self.d,):
            raise ValueError(
                f"boundary_mass must have shape ({self.d},), "
                f"got {tuple(boundary_mass_real.shape)}"
            )
        if not torch.isfinite(boundary_mass_real).all():
            raise ValueError("boundary_mass contains non-finite entries.")
        if torch.any(boundary_mass_real <= 0):
            raise ValueError(
                "boundary_mass entries must be strictly positive."
            )
        self.register_buffer(
            "boundary_mass",
            boundary_mass_real,
            persistent=True,
        )
        self.register_buffer(
            "boundary_log_mass",
            torch.log(boundary_mass_real).to(c_dtype),
            persistent=True,
        )

        scale = 0.01
        self.coord_embedding = nn.Parameter(
            scale * torch.complex(
                torch.randn(
                    self.d,
                    self.coord_embed_dim,
                    dtype=r_dtype,
                ),
                torch.randn(
                    self.d,
                    self.coord_embed_dim,
                    dtype=r_dtype,
                ),
            )
        )
        self.boundary_embedding = nn.Parameter(
            scale * torch.complex(
                torch.randn(
                    self.d,
                    self.boundary_embed_dim,
                    dtype=r_dtype,
                ),
                torch.randn(
                    self.d,
                    self.boundary_embed_dim,
                    dtype=r_dtype,
                ),
            )
        )
        self.interior_net = CoordinateCauchyScalarNet(
            1 + self.coord_embed_dim,
            interior_hidden_width,
        )
        self.boundary_net = CoordinateCauchyScalarNet(
            1 + self.coord_embed_dim + self.boundary_embed_dim,
            boundary_hidden_width,
        )

        face_indices = torch.arange(
            self.d,
            dtype=torch.long,
            device=boundary_mass_real.device,
        )
        coordinate_grid = face_indices.view(1, self.d).expand(
            self.d,
            self.d,
        )
        masks = coordinate_grid != face_indices.view(self.d, 1)
        other_indices = coordinate_grid[masks].view(
            self.d,
            self.d - 1,
        )
        self.register_buffer(
            "_face_indices",
            face_indices,
            persistent=False,
        )
        self.register_buffer(
            "_boundary_masks",
            masks,
            persistent=False,
        )
        self.register_buffer(
            "_other_coordinate_indices",
            other_indices,
            persistent=False,
        )

    @staticmethod
    def new_call_context():
        return {
            "interior_q0": None,
            "boundary_q0": None,
        }

    def _validate_theta(self, theta):
        theta = theta.to(dtype=c_dtype, device=device)
        if theta.ndim != 2 or theta.shape[1] != self.d:
            raise ValueError(
                f"theta must have shape [B,{self.d}], "
                f"got {tuple(theta.shape)}"
            )
        return theta

    def _face_blocks(self):
        for start in range(0, self.d, self.face_block_size):
            yield self._face_indices[
                start:min(start + self.face_block_size, self.d)
            ]

    def _interior_inputs(self, theta):
        batch_size, d = theta.shape
        coord = self.coord_embedding.view(
            1,
            d,
            self.coord_embed_dim,
        ).expand(
            batch_size,
            d,
            self.coord_embed_dim,
        )
        return torch.cat(
            [theta.unsqueeze(-1), coord],
            dim=2,
        ).reshape(
            batch_size * d,
            1 + self.coord_embed_dim,
        )

    def _interior_zero_outputs(self, theta, context=None):
        if context is not None and context.get("interior_q0") is not None:
            return context["interior_q0"]
        zero_theta = torch.zeros(
            self.d,
            1,
            dtype=c_dtype,
            device=theta.device,
        )
        x0 = torch.cat(
            [zero_theta, self.coord_embedding],
            dim=1,
        )
        q0 = run_scalar_net_chunked(
            self.interior_net,
            x0,
            self.pair_chunk_size,
            with_derivative=False,
        ).view(1, self.d)
        if context is not None:
            context["interior_q0"] = q0
        return q0

    def _interior_f0_and_jac(
        self,
        theta,
        need_jacobian,
        context=None,
    ):
        batch_size, d = theta.shape
        x = self._interior_inputs(theta)
        if need_jacobian:
            q, dq = run_scalar_net_chunked(
                self.interior_net,
                x,
                self.pair_chunk_size,
                with_derivative=True,
            )
            dq = dq.view(batch_size, d)
        else:
            q = run_scalar_net_chunked(
                self.interior_net,
                x,
                self.pair_chunk_size,
                with_derivative=False,
            )
            dq = None
        q = q.view(batch_size, d)
        q0 = self._interior_zero_outputs(theta, context=context)
        f0 = torch.sum(q - q0, dim=1, keepdim=True)
        return f0, dq

    def _boundary_zero_outputs(self, theta, context=None):
        if context is not None and context.get("boundary_q0") is not None:
            return context["boundary_q0"]
        other = self._other_coordinate_indices
        zeros = torch.zeros(
            self.d,
            self.d - 1,
            1,
            dtype=c_dtype,
            device=theta.device,
        )
        coord = self.coord_embedding[other]
        boundary = self.boundary_embedding.view(
            self.d,
            1,
            self.boundary_embed_dim,
        ).expand(
            self.d,
            self.d - 1,
            self.boundary_embed_dim,
        )
        x0 = torch.cat(
            [zeros, coord, boundary],
            dim=2,
        ).reshape(
            self.d * (self.d - 1),
            1 + self.coord_embed_dim + self.boundary_embed_dim,
        )
        q0 = run_scalar_net_chunked(
            self.boundary_net,
            x0,
            self.pair_chunk_size,
            with_derivative=False,
        ).view(self.d, self.d - 1)
        if context is not None:
            context["boundary_q0"] = q0
        return q0

    def _boundary_block_inputs(self, theta, faces):
        batch_size = theta.shape[0]
        face_count = faces.numel()
        other = self._other_coordinate_indices[faces]
        expanded_theta = theta.view(
            batch_size,
            1,
            self.d,
        ).expand(
            batch_size,
            face_count,
            self.d,
        )
        gather_index = other.view(
            1,
            face_count,
            self.d - 1,
        ).expand(
            batch_size,
            face_count,
            self.d - 1,
        )
        theta_selected = torch.gather(
            expanded_theta,
            dim=2,
            index=gather_index,
        )
        coord = self.coord_embedding[other].view(
            1,
            face_count,
            self.d - 1,
            self.coord_embed_dim,
        ).expand(
            batch_size,
            face_count,
            self.d - 1,
            self.coord_embed_dim,
        )
        boundary = self.boundary_embedding[faces].view(
            1,
            face_count,
            1,
            self.boundary_embed_dim,
        ).expand(
            batch_size,
            face_count,
            self.d - 1,
            self.boundary_embed_dim,
        )
        x = torch.cat(
            [theta_selected.unsqueeze(-1), coord, boundary],
            dim=3,
        )
        return x.reshape(
            batch_size * face_count * (self.d - 1),
            1 + self.coord_embed_dim + self.boundary_embed_dim,
        )

    def _boundary_paired_block_inputs(self, theta_faces, faces):
        batch_size, face_count, d = theta_faces.shape
        if d != self.d or face_count != faces.numel():
            raise ValueError("theta_faces and faces have incompatible shapes.")
        other = self._other_coordinate_indices[faces]
        gather_index = other.view(
            1,
            face_count,
            self.d - 1,
        ).expand(
            batch_size,
            face_count,
            self.d - 1,
        )
        theta_selected = torch.gather(
            theta_faces,
            dim=2,
            index=gather_index,
        )
        coord = self.coord_embedding[other].view(
            1,
            face_count,
            self.d - 1,
            self.coord_embed_dim,
        ).expand(
            batch_size,
            face_count,
            self.d - 1,
            self.coord_embed_dim,
        )
        boundary = self.boundary_embedding[faces].view(
            1,
            face_count,
            1,
            self.boundary_embed_dim,
        ).expand(
            batch_size,
            face_count,
            self.d - 1,
            self.boundary_embed_dim,
        )
        x = torch.cat(
            [theta_selected.unsqueeze(-1), coord, boundary],
            dim=3,
        )
        return x.reshape(
            batch_size * face_count * (self.d - 1),
            1 + self.coord_embed_dim + self.boundary_embed_dim,
        )

    def _boundary_block(
        self,
        theta,
        faces,
        need_derivative,
        context=None,
    ):
        batch_size = theta.shape[0]
        face_count = faces.numel()
        x = self._boundary_block_inputs(theta, faces)
        if need_derivative:
            q, dq = run_scalar_net_chunked(
                self.boundary_net,
                x,
                self.pair_chunk_size,
                with_derivative=True,
            )
            dq = dq.view(
                batch_size,
                face_count,
                self.d - 1,
            )
        else:
            q = run_scalar_net_chunked(
                self.boundary_net,
                x,
                self.pair_chunk_size,
                with_derivative=False,
            )
            dq = None
        q = q.view(
            batch_size,
            face_count,
            self.d - 1,
        )
        q0 = self._boundary_zero_outputs(
            theta,
            context=context,
        )[faces].view(
            1,
            face_count,
            self.d - 1,
        )
        fk = self.boundary_log_mass[faces].view(
            1,
            face_count,
        ) + torch.sum(q - q0, dim=2)
        return fk, dq

    def _boundary_paired_block(
        self,
        theta_faces,
        faces,
        context=None,
    ):
        batch_size, face_count, _ = theta_faces.shape
        x = self._boundary_paired_block_inputs(
            theta_faces,
            faces,
        )
        q = run_scalar_net_chunked(
            self.boundary_net,
            x,
            self.pair_chunk_size,
            with_derivative=False,
        ).view(
            batch_size,
            face_count,
            self.d - 1,
        )
        q0 = self._boundary_zero_outputs(
            theta_faces[:, 0, :],
            context=context,
        )[faces].view(
            1,
            face_count,
            self.d - 1,
        )
        fk = self.boundary_log_mass[faces].view(
            1,
            face_count,
        ) + torch.sum(q - q0, dim=2)
        return fk

    def forward_f0(self, theta, context=None):
        theta = self._validate_theta(theta)
        f0, _ = self._interior_f0_and_jac(
            theta,
            need_jacobian=False,
            context=context,
        )
        return f0

    def forward_selected_boundary(self, theta, k, context=None):
        theta = self._validate_theta(theta)
        if k < 0 or k >= self.d:
            raise IndexError(
                f"Boundary index k={k} outside [0,{self.d - 1}]"
            )
        faces = self._face_indices[k:k + 1]
        f0, _ = self._interior_f0_and_jac(
            theta,
            need_jacobian=False,
            context=context,
        )
        fk = self._boundary_paired_block(
            theta.unsqueeze(1),
            faces,
            context=context,
        )
        return f0, fk

    def forward_selected_boundary_block(
        self,
        theta_faces,
        faces,
        context=None,
    ):
        theta_faces = theta_faces.to(dtype=c_dtype, device=device)
        faces = faces.to(
            dtype=torch.long,
            device=theta_faces.device,
        )
        batch_size, face_count, d = theta_faces.shape
        if d != self.d or face_count != faces.numel():
            raise ValueError(
                "theta_faces must have shape [B,len(faces),d]."
            )
        flat_theta = theta_faces.reshape(
            batch_size * face_count,
            self.d,
        )
        f0, _ = self._interior_f0_and_jac(
            flat_theta,
            need_jacobian=False,
            context=context,
        )
        fk = self._boundary_paired_block(
            theta_faces,
            faces,
            context=context,
        )
        return f0.view(batch_size, face_count), fk

    def forward(self, theta, context=None):
        f_mat, _ = self.forward_with_input_jacobian(
            theta,
            need_jacobian=False,
            context=context,
        )
        return f_mat

    def forward_with_input_jacobian(
        self,
        theta,
        need_jacobian=True,
        context=None,
    ):
        theta = self._validate_theta(theta)
        batch_size = theta.shape[0]
        f0, dq_int = self._interior_f0_and_jac(
            theta,
            need_jacobian=need_jacobian,
            context=context,
        )
        boundary_blocks = []
        if need_jacobian:
            jacobian = torch.zeros(
                batch_size,
                self.d + 1,
                self.d,
                dtype=c_dtype,
                device=theta.device,
            )
            jacobian[:, 0, :] = dq_int
        else:
            jacobian = None

        for faces in self._face_blocks():
            fk, dq = self._boundary_block(
                theta,
                faces,
                need_derivative=need_jacobian,
                context=context,
            )
            boundary_blocks.append(fk)
            if need_jacobian:
                face_count = faces.numel()
                block_jacobian = torch.zeros(
                    batch_size,
                    face_count,
                    self.d,
                    dtype=c_dtype,
                    device=theta.device,
                )
                scatter_index = self._other_coordinate_indices[
                    faces
                ].view(
                    1,
                    face_count,
                    self.d - 1,
                ).expand(
                    batch_size,
                    face_count,
                    self.d - 1,
                )
                block_jacobian.scatter_(
                    dim=2,
                    index=scatter_index,
                    src=dq,
                )
                jacobian[:, faces + 1, :] = block_jacobian

        boundary_values = torch.cat(boundary_blocks, dim=1)
        return torch.cat([f0, boundary_values], dim=1), jacobian


## 5. BAR objective and memory-bounded backward pass

BAR, boundary, and monotonicity/imaginary branches are differentiated
sequentially before one Adam update. This preserves the summed gradient while
reducing peak accelerator memory. It does not change the loss or batch budget.


In [ ]:
# BAR, boundary, analytical monotonicity, and sequential backward

DEFAULT_ADAPTIVE_BAR_CONFIG = {
    "adaptive_bar_weight": True,
    "bar_weight_alpha": 0.3,
    "bar_weight_eps": 1e-12,
    "bar_weight_min": 0.5,
    "bar_weight_max": 2.0,
}

def with_adaptive_bar_defaults(config):
    merged = dict(DEFAULT_ADAPTIVE_BAR_CONFIG)
    merged.update(config)
    return merged

def adaptive_bar_kwargs(config):
    cfg = with_adaptive_bar_defaults(config)
    return {
        "adaptive_bar_weight": bool(cfg["adaptive_bar_weight"]),
        "bar_weight_alpha": float(cfg["bar_weight_alpha"]),
        "bar_weight_eps": float(cfg["bar_weight_eps"]),
        "bar_weight_min": float(cfg["bar_weight_min"]),
        "bar_weight_max": float(cfg["bar_weight_max"]),
    }

def split_f_tuple(f_out):
    if isinstance(f_out, torch.Tensor):
        return f_out
    if isinstance(f_out, (tuple, list)):
        return torch.cat(list(f_out), dim=1)
    raise TypeError(f"Unsupported model output type {type(f_out)!r}")

def compute_bar_pointwise(theta, f_tuple, params, eps_gamma=1e-12, eps_bar=1e-8):
    f_mat = split_f_tuple(f_tuple)
    gamma0, gamma_all = compute_gammas(theta, params)
    gamma_mat = torch.cat([gamma0, gamma_all], dim=1)
    nu = torch.max(torch.log(torch.abs(gamma_mat) + eps_gamma) + f_mat.real, dim=1, keepdim=True)[0]
    kappa = gamma_mat * torch.exp(f_mat - nu.to(dtype=c_dtype))
    left = kappa[:, 0:1]
    right = torch.sum(kappa[:, 1:], dim=1, keepdim=True)
    return (torch.abs(left - right) / (torch.abs(left) + torch.abs(right) + eps_bar)) ** 2

def weighted_normalized_mean(pointwise, adaptive_bar_weight=True, bar_weight_alpha=0.3, bar_weight_eps=1e-12, bar_weight_min=0.5, bar_weight_max=2.0):
    unweighted = torch.mean(pointwise)
    if adaptive_bar_weight:
        raw_w = (pointwise.detach() + bar_weight_eps) ** bar_weight_alpha
        weights = torch.clamp(raw_w / (torch.mean(raw_w) + bar_weight_eps), min=bar_weight_min, max=bar_weight_max).detach()
        weighted = torch.sum(weights * pointwise) / (torch.sum(weights) + bar_weight_eps)
    else:
        weights = torch.ones_like(pointwise).detach()
        weighted = unweighted
    return weighted, unweighted, weights

def compute_bar_loss(theta, f_tuple, params, eps_gamma=1e-12, eps_bar=1e-8, adaptive_bar_weight=True, bar_weight_alpha=0.3, bar_weight_eps=1e-12, bar_weight_min=0.5, bar_weight_max=2.0, return_diagnostics=False):
    pointwise = compute_bar_pointwise(theta, f_tuple, params, eps_gamma=eps_gamma, eps_bar=eps_bar)
    weighted, unweighted, weights = weighted_normalized_mean(
        pointwise,
        adaptive_bar_weight=adaptive_bar_weight,
        bar_weight_alpha=bar_weight_alpha,
        bar_weight_eps=bar_weight_eps,
        bar_weight_min=bar_weight_min,
        bar_weight_max=bar_weight_max,
    )
    if not return_diagnostics:
        return weighted
    return weighted, {
        "bar_unweighted": unweighted.detach(),
        "bar_weighted": weighted.detach(),
        "bar_weight_mean": torch.mean(weights).detach(),
        "bar_weight_min": torch.min(weights).detach(),
        "bar_weight_max": torch.max(weights).detach(),
    }

def boundary_face_residual_from_f(theta_k, f0, fk, params, k, eps_gamma=1e-12, eps_bar=1e-8):
    gamma0, gamma_all = compute_gammas(theta_k, params)
    gammak = gamma_all[:, k:k+1]
    nu = torch.maximum(torch.log(torch.abs(gamma0) + eps_gamma) + f0.real, torch.log(torch.abs(gammak) + eps_gamma) + fk.real).to(dtype=c_dtype)
    left = gamma0 * torch.exp(f0 - nu)
    right = gammak * torch.exp(fk - nu)
    rel = torch.abs(left - right) / (torch.abs(left) + torch.abs(right) + eps_bar)
    return rel ** 2

def compute_zero_loss(model, params):
    theta0 = torch.zeros(1, params.R.shape[0], dtype=c_dtype, device=device)
    return torch.mean(torch.abs(torch.exp(model(theta0)[:, 0:1]) - 1.0) ** 2)

def create_heldout_batches(config):
    cfg = with_adaptive_bar_defaults(config)
    seed = int(cfg.get("seed", 12345))
    eval_seed = int(cfg.get("eval_seed", seed + 1000003))
    d = cfg["d"]
    eval_batch_bar = int(cfg.get("eval_batch_bar", cfg["batch_bar"]))
    eval_batch_bdary = int(cfg.get("eval_batch_bdary", cfg["batch_bdary"]))
    eval_batch_mono = int(cfg.get("eval_batch_mono", cfg["batch_mono"]))
    devices = [torch.cuda.current_device()] if torch.cuda.is_available() else []
    with torch.random.fork_rng(devices=devices, enabled=True):
        torch.manual_seed(eval_seed)
        if torch.cuda.is_available():
            torch.cuda.manual_seed_all(eval_seed)
        theta_bar_eval = sample_mixed_bar_theta(N=eval_batch_bar, d=d, re_min=-0.5, re_max=5.0, im_max=5.0, device=device).detach().clone()
        theta_bdary_eval = sample_training_theta(eval_batch_bdary, d=d, re_min=-0.5, re_max=5.0, im_max=5.0, device=device).detach().clone()
        theta_mono = sample_training_theta(eval_batch_mono, d=d, re_min=cfg["mono_re_min"], re_max=cfg["mono_re_max"], im_max=0.0, device=device)
        theta_mono_real_eval = theta_mono.real.detach().clone()
    return {"theta_bar_eval": theta_bar_eval, "theta_bdary_eval": theta_bdary_eval, "theta_mono_real_eval": theta_mono_real_eval, "eval_seed": eval_seed}

def _boundary_face_block_losses(
    model,
    theta,
    params,
    eps_gamma=1e-12,
    eps_bar=1e-8,
    boundary_directions=None,
    context=None,
):
    theta = theta.to(dtype=c_dtype, device=device)
    batch_size, d = theta.shape
    directions = (
        boundary_directions
        if boundary_directions is not None
        else get_boundary_directions_cached(
            params,
            target_device=theta.device,
            target_dtype=theta.dtype,
        )
    )
    face_losses = []
    for faces in model._face_blocks():
        face_count = faces.numel()
        theta_faces = (
            theta[:, faces].unsqueeze(-1)
            * directions[faces, :].view(1, face_count, d)
        )
        f0, fk = model.forward_selected_boundary_block(
            theta_faces,
            faces,
            context=context,
        )
        flat_theta = theta_faces.reshape(batch_size * face_count, d)
        gamma0, gamma_all = compute_gammas(flat_theta, params)
        gamma0 = gamma0.view(batch_size, face_count)
        gamma_all = gamma_all.view(batch_size, face_count, d)
        matching_index = faces.view(
            1,
            face_count,
            1,
        ).expand(
            batch_size,
            face_count,
            1,
        )
        gamma_k = torch.gather(
            gamma_all,
            dim=2,
            index=matching_index,
        ).squeeze(2)
        nu = torch.maximum(
            torch.log(torch.abs(gamma0) + eps_gamma) + f0.real,
            torch.log(torch.abs(gamma_k) + eps_gamma) + fk.real,
        )
        left = gamma0 * torch.exp(f0 - nu.to(c_dtype))
        right = gamma_k * torch.exp(fk - nu.to(c_dtype))
        residual = torch.abs(left - right) / (
            torch.abs(left) + torch.abs(right) + eps_bar
        )
        face_losses.append(torch.mean(residual ** 2, dim=0))
    return torch.cat(face_losses, dim=0)

def compute_boundary_multiplicative_loss_structured(
    model,
    theta,
    params,
    eps_gamma=1e-12,
    eps_bar=1e-8,
    boundary_directions=None,
    context=None,
):
    face_losses = _boundary_face_block_losses(
        model,
        theta,
        params,
        eps_gamma=eps_gamma,
        eps_bar=eps_bar,
        boundary_directions=boundary_directions,
        context=context,
    )
    return torch.sum(face_losses) / face_losses.numel()

def compute_monotonicity_components_analytical_from_theta_real(
    model,
    theta_real,
    context=None,
):
    theta_real = theta_real.to(dtype=r_dtype, device=device)
    theta = torch.complex(
        theta_real,
        torch.zeros_like(theta_real),
    )
    f_mat, jacobian = model.forward_with_input_jacobian(
        theta,
        need_jacobian=True,
        context=context,
    )
    loss_mono = torch.mean(torch.relu(jacobian.real) ** 2)
    loss_imag = torch.mean(f_mat.imag ** 2)
    return loss_mono, loss_imag

def _profile_start(profiler, name):
    if profiler is not None:
        profiler.start_section(name)

def _profile_end(profiler, name):
    if profiler is not None:
        profiler.end_section(name)

def _require_finite_loss(name, value):
    if not torch.isfinite(value).all():
        raise FloatingPointError(f"Non-finite {name}.")

def compute_parameter_gradient_norm(model):
    squared_norms = [
        torch.sum(torch.abs(parameter.grad) ** 2)
        for parameter in model.parameters()
        if parameter.grad is not None
    ]
    if not squared_norms:
        return torch.zeros((), dtype=r_dtype, device=device)
    return torch.sqrt(torch.sum(torch.stack(squared_norms)))

def sample_sequential_training_inputs(config):
    # Keep the original RNG order: BAR, boundary, then monotonicity.
    theta_bar = sample_mixed_bar_theta(
        N=config["batch_bar"],
        d=config["d"],
        re_min=-0.5,
        re_max=5.0,
        im_max=5.0,
        device=device,
    )
    theta_bdary = sample_training_theta(
        config["batch_bdary"],
        d=config["d"],
        re_min=-0.5,
        re_max=5.0,
        im_max=5.0,
        device=device,
    )
    theta_mono = sample_training_theta(
        config["batch_mono"],
        d=config["d"],
        re_min=config["mono_re_min"],
        re_max=config["mono_re_max"],
        im_max=0.0,
        device=device,
    )
    return {
        "theta_bar": theta_bar,
        "theta_bdary": theta_bdary,
        "theta_mono_real": theta_mono.real.detach(),
    }

def compute_bar_branch_from_theta(model, params, theta_bar, config):
    context = model.new_call_context()
    f_bar = model(theta_bar, context=context)
    loss_bar, bar_diag = compute_bar_loss(
        theta_bar,
        f_bar,
        params,
        **adaptive_bar_kwargs(config),
        return_diagnostics=True,
    )
    weighted = config["lambda_bar"] * loss_bar
    _require_finite_loss("BAR loss", loss_bar)
    _require_finite_loss("weighted BAR branch", weighted)
    return {
        "loss_bar": loss_bar,
        "weighted": weighted,
        "bar_diag": bar_diag,
        "f_bar": f_bar,
    }, context

def compute_boundary_branch_from_theta(
    model,
    params,
    theta_bdary,
    config,
):
    context = model.new_call_context()
    loss_bdary = compute_boundary_multiplicative_loss_structured(
        model,
        theta_bdary,
        params,
        boundary_directions=get_boundary_directions_cached(
            params,
            target_device=theta_bdary.device,
            target_dtype=theta_bdary.dtype,
        ),
        context=context,
    )
    weighted = config["lambda_bdary"] * loss_bdary
    _require_finite_loss("boundary loss", loss_bdary)
    _require_finite_loss("weighted boundary branch", weighted)
    return {
        "loss_bdary": loss_bdary,
        "weighted": weighted,
    }, context

def compute_mono_imag_branch_from_theta(
    model,
    params,
    theta_mono_real,
    config,
):
    context = model.new_call_context()
    if config["lambda_zero"] == 0.0:
        loss_zero = torch.zeros(
            (),
            dtype=r_dtype,
            device=device,
        )
    else:
        theta0 = torch.zeros(
            1,
            params.R.shape[0],
            dtype=c_dtype,
            device=device,
        )
        f0_at_zero = model(theta0, context=context)[:, 0:1]
        loss_zero = torch.mean(
            torch.abs(torch.exp(f0_at_zero) - 1.0) ** 2
        )
    loss_mono, loss_imag = (
        compute_monotonicity_components_analytical_from_theta_real(
            model,
            theta_mono_real,
            context=context,
        )
    )
    weighted = (
        config["lambda_zero"] * loss_zero
        + config["lambda_mono"] * loss_mono
        + config["lambda_imag"] * loss_imag
    )
    _require_finite_loss("zero loss", loss_zero)
    _require_finite_loss("monotonicity loss", loss_mono)
    _require_finite_loss("imaginary loss", loss_imag)
    _require_finite_loss("weighted mono/imag branch", weighted)
    return {
        "loss_zero": loss_zero,
        "loss_mono": loss_mono,
        "loss_imag": loss_imag,
        "weighted": weighted,
    }, context

def run_sequential_backward_training_step(
    model,
    optimizer,
    params,
    config,
    profiler=None,
    fixed_inputs=None,
    clip_gradients=True,
    perform_optimizer_step=True,
):
    optimizer.zero_grad(set_to_none=True)

    _profile_start(profiler, "sampling")
    if fixed_inputs is None:
        inputs = sample_sequential_training_inputs(config)
    else:
        inputs = {
            "theta_bar": fixed_inputs["theta_bar"],
            "theta_bdary": fixed_inputs["theta_bdary"],
            "theta_mono_real": fixed_inputs["theta_mono_real"],
        }
    _profile_end(profiler, "sampling")

    _profile_start(profiler, "bar_forward")
    bar_branch, bar_context = compute_bar_branch_from_theta(
        model,
        params,
        inputs["theta_bar"],
        config,
    )
    bar_loss_detached = bar_branch["loss_bar"].detach()
    weighted_bar_detached = bar_branch["weighted"].detach()
    bar_diag_detached = {
        key: value.detach() if isinstance(value, torch.Tensor) else value
        for key, value in bar_branch["bar_diag"].items()
    }
    _profile_end(profiler, "bar_forward")
    _profile_start(profiler, "bar_backward")
    if config["lambda_bar"] != 0.0:
        bar_branch["weighted"].backward()
    _profile_end(profiler, "bar_backward")
    del bar_branch, bar_context

    _profile_start(profiler, "boundary_forward")
    boundary_branch, boundary_context = (
        compute_boundary_branch_from_theta(
            model,
            params,
            inputs["theta_bdary"],
            config,
        )
    )
    boundary_loss_detached = boundary_branch["loss_bdary"].detach()
    weighted_boundary_detached = boundary_branch["weighted"].detach()
    _profile_end(profiler, "boundary_forward")
    _profile_start(profiler, "boundary_backward")
    if config["lambda_bdary"] != 0.0:
        boundary_branch["weighted"].backward()
    _profile_end(profiler, "boundary_backward")
    del boundary_branch, boundary_context

    _profile_start(profiler, "monotonicity_imaginary_forward")
    mono_branch, mono_context = compute_mono_imag_branch_from_theta(
        model,
        params,
        inputs["theta_mono_real"],
        config,
    )
    zero_loss_detached = mono_branch["loss_zero"].detach()
    mono_loss_detached = mono_branch["loss_mono"].detach()
    imag_loss_detached = mono_branch["loss_imag"].detach()
    weighted_mono_imag_detached = mono_branch["weighted"].detach()
    _profile_end(profiler, "monotonicity_imaginary_forward")
    _profile_start(profiler, "monotonicity_imaginary_backward")
    mono_branch_active = any(
        config[key] != 0.0
        for key in ("lambda_zero", "lambda_mono", "lambda_imag")
    )
    if mono_branch_active:
        mono_branch["weighted"].backward()
    _profile_end(profiler, "monotonicity_imaginary_backward")
    del mono_branch, mono_context, inputs

    total_loss_detached = (
        weighted_bar_detached
        + weighted_boundary_detached
        + weighted_mono_imag_detached
    )
    _require_finite_loss("detached total loss", total_loss_detached)
    diagnostics = {
        "total_loss": total_loss_detached,
        "loss_bar": bar_loss_detached,
        "loss_bar_unweighted": bar_diag_detached["bar_unweighted"],
        "loss_bar_weighted": bar_diag_detached["bar_weighted"],
        "loss_bdary": boundary_loss_detached,
        "loss_zero": zero_loss_detached,
        "loss_mono": mono_loss_detached,
        "loss_imag": imag_loss_detached,
    }
    diagnostics.update(bar_diag_detached)

    for name, parameter in model.named_parameters():
        if (
            parameter.grad is not None
            and not torch.isfinite(parameter.grad).all()
        ):
            raise FloatingPointError(f"Non-finite gradient for {name}.")

    _profile_start(profiler, "gradient_clipping")
    if clip_gradients:
        grad_norm = torch.nn.utils.clip_grad_norm_(
            model.parameters(),
            config["grad_clip"],
        )
    else:
        grad_norm = compute_parameter_gradient_norm(model)
    _profile_end(profiler, "gradient_clipping")
    _require_finite_loss("gradient norm", grad_norm)

    _profile_start(profiler, "optimizer_step")
    if perform_optimizer_step:
        optimizer.step()
    _profile_end(profiler, "optimizer_step")
    return total_loss_detached, diagnostics, grad_norm


## 6. Preflight, smoke test, and fixed-endpoint training

Formal training always starts from the prespecified seed or an exact recovery
checkpoint produced by this notebook. Recovery files protect against interruption
but are never evaluated or used to select a model. The only reported checkpoint
is step 100,000.


In [ ]:
LOG_COLUMNS = [
    "seed", "step", "learning_rate", "total_loss", "bar_loss",
    "bar_loss_unweighted", "boundary_loss", "monotonicity_loss",
    "imaginary_loss", "gradient_norm", "minimum_cauchy_denominator",
    "cumulative_sampled_constraints", "elapsed_seconds",
]


def build_model(seed=PAPER_SEED):
    set_all_seeds(seed)
    return StructuredCoordinateCauchyMGF(
        d=MODEL_CONFIG["d"],
        coord_embed_dim=MODEL_CONFIG["coord_embed_dim"],
        boundary_embed_dim=MODEL_CONFIG["boundary_embed_dim"],
        interior_hidden_width=MODEL_CONFIG["interior_hidden_width"],
        boundary_hidden_width=MODEL_CONFIG["boundary_hidden_width"],
        pair_chunk_size=MODEL_CONFIG["pair_chunk_size"],
        boundary_mass=BOUNDARY_MASS,
        face_block_size=MODEL_CONFIG["face_block_size"],
    ).to(device)


def parameter_audit(model):
    complex_parameters = sum(
        parameter.numel() for parameter in model.parameters()
        if parameter.is_complex()
    )
    real_parameters = sum(
        parameter.numel() for parameter in model.parameters()
        if not parameter.is_complex()
    )
    return {
        "complex_parameter_count": int(complex_parameters),
        "real_parameter_count": int(real_parameters),
        "trainable_real_scalar_dof": int(2 * complex_parameters + real_parameters),
    }


PARAMETER_AUDIT = parameter_audit(build_model())
if PARAMETER_AUDIT["trainable_real_scalar_dof"] != 18_308:
    raise AssertionError(PARAMETER_AUDIT)


def reset_denominator_tracking(model):
    for module in model.modules():
        if isinstance(module, CauchyComplex):
            module.reset_denominator_tracking()


def minimum_cauchy_denominator(model):
    values = [
        module.running_min_abs_denominator
        for module in model.modules()
        if isinstance(module, CauchyComplex)
        and module.running_min_abs_denominator is not None
    ]
    if not values:
        return float("nan")
    return float(torch.min(torch.stack(values)).detach().cpu())


def gradients_are_finite(model):
    gradients = [p.grad for p in model.parameters() if p.requires_grad]
    return bool(
        gradients
        and all(g is not None and torch.isfinite(g).all().item() for g in gradients)
    )


def run_preflight():
    model = build_model()
    model.train()
    theta = sample_training_theta(4, d=DIMENSION, device=device)
    values, jacobian = model.forward_with_input_jacobian(theta, need_jacobian=True)
    if values.shape != (4, DIMENSION + 1):
        raise AssertionError(f"Unexpected output shape {tuple(values.shape)}")
    if jacobian.shape != (4, DIMENSION + 1, DIMENSION):
        raise AssertionError(f"Unexpected Jacobian shape {tuple(jacobian.shape)}")

    zero = torch.zeros(2, DIMENSION, dtype=c_dtype, device=device)
    with torch.no_grad():
        at_zero = model(zero)
    interior_origin_error = float(torch.max(torch.abs(at_zero[:, 0])).cpu())
    boundary_anchor_error = float(torch.max(
        torch.abs(at_zero[:, 1:] - torch.log(BOUNDARY_MASS).to(c_dtype))
    ).cpu())

    optimizer = torch.optim.Adam(model.parameters(), lr=TRAIN_CONFIG["lr_stage1"])
    tiny = dict(TRAIN_CONFIG)
    tiny.update({"batch_bar": 16, "batch_bdary": 4, "batch_mono": 4})
    reset_denominator_tracking(model)
    total, diagnostics, gradient_norm = run_sequential_backward_training_step(
        model, optimizer, PARAMS, tiny, perform_optimizer_step=False
    )
    report = {
        "seed": PAPER_SEED,
        "output_shape": list(values.shape),
        "jacobian_shape": list(jacobian.shape),
        "jacobian_finite": bool(torch.isfinite(jacobian).all().item()),
        "loss_terms_finite": bool(all(
            torch.isfinite(value).all().item()
            for value in diagnostics.values() if torch.is_tensor(value)
        )),
        "gradients_present_and_finite": gradients_are_finite(model),
        "gradient_norm": float(gradient_norm.detach().cpu()),
        "interior_origin_error": interior_origin_error,
        "boundary_anchor_error": boundary_anchor_error,
        "minimum_cauchy_denominator": minimum_cauchy_denominator(model),
        "parameter_real_scalar_dof": PARAMETER_AUDIT["trainable_real_scalar_dof"],
    }
    if (
        not report["jacobian_finite"]
        or not report["loss_terms_finite"]
        or not report["gradients_present_and_finite"]
        or interior_origin_error > 1.0e-11
        or boundary_anchor_error > 1.0e-11
        or not math.isfinite(report["minimum_cauchy_denominator"])
        or report["minimum_cauchy_denominator"] <= 0.0
    ):
        raise AssertionError(report)
    return report


def run_smoke_test():
    preflight = run_preflight()
    model = build_model()
    optimizer = torch.optim.Adam(model.parameters(), lr=TRAIN_CONFIG["lr_stage1"])
    tiny = dict(TRAIN_CONFIG)
    tiny.update({"batch_bar": 16, "batch_bdary": 4, "batch_mono": 4})
    losses = []
    for _ in range(2):
        reset_denominator_tracking(model)
        total, _, _ = run_sequential_backward_training_step(
            model, optimizer, PARAMS, tiny
        )
        losses.append(float(total.detach().cpu()))
    report = {"preflight": preflight, "two_step_losses": losses, "passed": True}
    print(json.dumps(report, indent=2))
    return report


def learning_rate_for_step(step):
    return (
        TRAIN_CONFIG["lr_stage1"]
        if int(step) <= LR_SWITCH_AFTER_STEP
        else TRAIN_CONFIG["lr_stage2"]
    )


def set_optimizer_learning_rate(optimizer, value):
    for group in optimizer.param_groups:
        group["lr"] = float(value)


def optimizer_to_device(optimizer):
    for state in optimizer.state.values():
        for key, value in state.items():
            if torch.is_tensor(value):
                state[key] = value.to(device)


def checkpoint_payload(model, optimizer, completed_step):
    return {
        "schema": "hcn_d50_fixed100k_v1",
        "experiment_slug": EXPERIMENT_SLUG,
        "seed": PAPER_SEED,
        "last_completed_step": int(completed_step),
        "fixed_endpoint_step": CHECKPOINT_STEP,
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "model_config": MODEL_CONFIG,
        "training_config": TRAIN_CONFIG,
        "sampler_config": SAMPLER_CONFIG,
        "parameter_audit": PARAMETER_AUDIT,
        "sample_budget_at_completed_step": int(completed_step) * (
            TRAIN_CONFIG["batch_bar"]
            + TRAIN_CONFIG["batch_bdary"]
            + TRAIN_CONFIG["batch_mono"]
        ),
        "torch_cpu_rng_state": torch.get_rng_state(),
        "torch_cuda_rng_state_all": (
            torch.cuda.get_rng_state_all() if torch.cuda.is_available() else []
        ),
        "boundary_mass": BOUNDARY_MASS.detach().cpu(),
    }


def atomic_torch_save(payload, path):
    path = Path(path)
    temporary = path.with_name(path.name + ".tmp")
    torch.save(payload, temporary)
    os.replace(temporary, path)


def validate_resume(payload):
    if payload.get("schema") != "hcn_d50_fixed100k_v1":
        raise RuntimeError("This is not a recovery checkpoint from this notebook.")
    if payload.get("experiment_slug") != EXPERIMENT_SLUG:
        raise RuntimeError("Recovery experiment mismatch.")
    if int(payload.get("seed", -1)) != PAPER_SEED:
        raise RuntimeError("Recovery seed mismatch.")
    if payload.get("model_config") != MODEL_CONFIG:
        raise RuntimeError("Recovery model configuration mismatch.")
    if payload.get("training_config") != TRAIN_CONFIG:
        raise RuntimeError("Recovery training configuration mismatch.")
    if payload.get("sampler_config") != SAMPLER_CONFIG:
        raise RuntimeError("Recovery sampler configuration mismatch.")
    completed = int(payload.get("last_completed_step", -1))
    if completed < 0 or completed >= TRAIN_STEPS:
        raise RuntimeError(f"Invalid recovery step {completed}.")
    return completed


def restore_training_state(path):
    path = Path(path).expanduser().resolve()
    if not path.is_file():
        raise FileNotFoundError(path)
    payload = torch.load(path, map_location=device, weights_only=True)
    completed = validate_resume(payload)
    model = build_model()
    model.load_state_dict(payload["model_state_dict"], strict=True)
    optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate_for_step(completed + 1))
    optimizer.load_state_dict(payload["optimizer_state_dict"])
    optimizer_to_device(optimizer)
    torch.set_rng_state(payload["torch_cpu_rng_state"].cpu())
    if torch.cuda.is_available():
        states = payload["torch_cuda_rng_state_all"]
        if len(states) != torch.cuda.device_count():
            raise RuntimeError("CUDA device count differs from the recovery checkpoint.")
        torch.cuda.set_rng_state_all([state.cpu() for state in states])
    return model, optimizer, completed, path.parent.parent


def read_training_log(path):
    path = Path(path)
    if not path.is_file():
        return []
    integer_columns = {"seed", "step", "cumulative_sampled_constraints"}
    rows = []
    with path.open("r", newline="", encoding="utf-8") as handle:
        for raw in csv.DictReader(handle):
            rows.append({
                key: int(value) if key in integer_columns else float(value)
                for key, value in raw.items()
            })
    return rows


def save_training_plot(rows, path):
    steps = [row["step"] for row in rows]
    plt.figure(figsize=(9.0, 5.5))
    plt.semilogy(steps, [row["total_loss"] for row in rows], label="total")
    plt.semilogy(steps, [row["bar_loss"] for row in rows], label="BAR")
    plt.semilogy(steps, [row["boundary_loss"] for row in rows], label="boundary")
    plt.axvline(
        LR_SWITCH_AFTER_STEP, color="black", linestyle="--",
        label="learning-rate switch",
    )
    plt.xlabel("optimizer step")
    plt.ylabel("loss")
    plt.grid(True, which="both", alpha=0.3)
    plt.legend()
    plt.tight_layout()
    plt.savefig(path, dpi=180)
    plt.close()


def initialize_formal_run(preflight):
    OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
    run_dir = OUTPUT_ROOT / f"{EXPERIMENT_SLUG}_{utc_stamp()}"
    run_dir.mkdir(parents=False, exist_ok=False)
    seed_dir = run_dir / f"seed_{PAPER_SEED}"
    seed_dir.mkdir(parents=False, exist_ok=False)
    write_json(run_dir / "run_config.json", RUN_CONFIG)
    write_json(run_dir / "rbm_audit.json", RBM_AUDIT)
    write_json(run_dir / "parameter_budget_audit.json", PARAMETER_AUDIT)
    write_json(run_dir / "sample_budget.json", SAMPLE_BUDGET)
    write_json(run_dir / "sampler_audit.json", {
        "configuration": SAMPLER_CONFIG,
        "actual_bar_counts_per_update": {
            "general_complex": 256,
            "exact_complex_diagonal": 102,
            "near_complex_diagonal": 102,
            "positive_real_diagonal": 52,
        },
    })
    write_json(run_dir / "preflight_audit.json", preflight)
    write_json(run_dir / "environment.json", {
        "python": sys.version,
        "platform": platform.platform(),
        "torch": torch.__version__,
        "cuda_version": torch.version.cuda,
        "cuda_available": torch.cuda.is_available(),
        "gpu_name": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
        "deterministic_algorithms": torch.are_deterministic_algorithms_enabled(),
        "timestamp_utc": datetime.now(timezone.utc).isoformat(),
    })
    return run_dir


def train_fixed_endpoint(resume_checkpoint=None):
    preflight = run_preflight()
    if resume_checkpoint is None:
        run_dir = initialize_formal_run(preflight)
        seed_dir = run_dir / f"seed_{PAPER_SEED}"
        model = build_model()
        optimizer = torch.optim.Adam(
            model.parameters(), lr=TRAIN_CONFIG["lr_stage1"]
        )
        completed_step = 0
        rows = []
    else:
        model, optimizer, completed_step, run_dir = restore_training_state(
            resume_checkpoint
        )
        seed_dir = run_dir / f"seed_{PAPER_SEED}"
        if Path(resume_checkpoint).expanduser().resolve().parent != seed_dir.resolve():
            raise RuntimeError("Recovery checkpoint is outside its seed directory.")
        rows = read_training_log(seed_dir / "training_log.csv")
        print(f"Resuming exact state after step {completed_step} from {resume_checkpoint}")

    elapsed_offset = max(
        (float(row["elapsed_seconds"]) for row in rows),
        default=0.0,
    )

    model.train()
    if torch.cuda.is_available():
        torch.cuda.reset_peak_memory_stats()
        torch.cuda.synchronize()
    started = time.perf_counter()

    for step in range(completed_step + 1, TRAIN_STEPS + 1):
        required_lr = learning_rate_for_step(step)
        if optimizer.param_groups[0]["lr"] != required_lr:
            set_optimizer_learning_rate(optimizer, required_lr)

        reset_denominator_tracking(model)
        total, diagnostics, gradient_norm = run_sequential_backward_training_step(
            model, optimizer, PARAMS, TRAIN_CONFIG
        )

        should_log = step % LOG_EVERY == 0 or step == TRAIN_STEPS
        if should_log:
            if torch.cuda.is_available():
                torch.cuda.synchronize()
            row = {
                "seed": PAPER_SEED,
                "step": step,
                "learning_rate": optimizer.param_groups[0]["lr"],
                "total_loss": float(total.detach().cpu()),
                "bar_loss": float(diagnostics["loss_bar"].detach().cpu()),
                "bar_loss_unweighted": float(
                    diagnostics["loss_bar_unweighted"].detach().cpu()
                ),
                "boundary_loss": float(diagnostics["loss_bdary"].detach().cpu()),
                "monotonicity_loss": float(diagnostics["loss_mono"].detach().cpu()),
                "imaginary_loss": float(diagnostics["loss_imag"].detach().cpu()),
                "gradient_norm": float(gradient_norm.detach().cpu()),
                "minimum_cauchy_denominator": minimum_cauchy_denominator(model),
                "cumulative_sampled_constraints": step * (
                    TRAIN_CONFIG["batch_bar"]
                    + TRAIN_CONFIG["batch_bdary"]
                    + TRAIN_CONFIG["batch_mono"]
                ),
                "elapsed_seconds": elapsed_offset + time.perf_counter() - started,
            }
            rows = [old for old in rows if int(old["step"]) != step]
            rows.append(row)
            rows.sort(key=lambda item: int(item["step"]))
            write_csv(seed_dir / "training_log.csv", rows, LOG_COLUMNS)
            print(row, flush=True)

        if step % RECOVERY_EVERY == 0 and step < TRAIN_STEPS:
            atomic_torch_save(
                checkpoint_payload(model, optimizer, step),
                seed_dir / "resume.pt",
            )
            if step == LR_SWITCH_AFTER_STEP:
                atomic_torch_save(
                    checkpoint_payload(model, optimizer, step),
                    seed_dir / f"milestone_step{step}.pt",
                )

    if torch.cuda.is_available():
        torch.cuda.synchronize()
    elapsed = time.perf_counter() - started
    final_checkpoint = seed_dir / f"checkpoint_step{CHECKPOINT_STEP}.pt"
    atomic_torch_save(
        checkpoint_payload(model, optimizer, TRAIN_STEPS), final_checkpoint
    )
    final_hash = sha256_file(final_checkpoint)
    save_training_plot(rows, seed_dir / "training_loss.png")
    write_json(seed_dir / "checkpoint_sha256.json", {
        "file": final_checkpoint.name,
        "sha256": final_hash,
        "step": CHECKPOINT_STEP,
        "seed": PAPER_SEED,
    })
    write_json(seed_dir / "training_runtime.json", {
        "session_start_completed_step": completed_step,
        "session_wall_clock_seconds": elapsed,
        "total_logged_wall_clock_seconds": elapsed_offset + elapsed,
        "peak_gpu_allocated_bytes": (
            int(torch.cuda.max_memory_allocated()) if torch.cuda.is_available() else 0
        ),
        "peak_gpu_reserved_bytes": (
            int(torch.cuda.max_memory_reserved()) if torch.cuda.is_available() else 0
        ),
        "checkpoint": str(final_checkpoint),
        "checkpoint_sha256": final_hash,
    })
    print("Fixed step-100,000 endpoint written to", final_checkpoint)
    return run_dir


SMOKE_REPORT = None
FORMAL_RUN_DIR = None
if RUN_MODE == "smoke":
    SMOKE_REPORT = run_smoke_test()
elif RUN_MODE == "train":
    FORMAL_RUN_DIR = train_fixed_endpoint(RESUME_CHECKPOINT)


## 7. Fixed-endpoint Talbot evaluation

Ground truth is deliberately introduced only here, after all training code.
Evaluation verifies the fixed step and configuration, hashes the checkpoint
before and after use, reports raw predictions without clipping, and records
every invalid or nonfinite outcome. Degree 8 is primary; Degree 5 is retained
as the historical numerical diagnostic used by the original 50D run.


In [ ]:
mp.mp.dps = EVALUATION_CONFIG["mpmath_decimal_digits"]
T_VALUES = EVALUATION_CONFIG["t_values"]
TALBOT_DEGREES = EVALUATION_CONFIG["talbot_degrees"]
RATES = [mp.mpf(index) for index in range(1, DIMENSION + 1)]


def exact_product_form_transform(s):
    s = mp.mpc(s)
    value = mp.mpf("1")
    for rate in RATES:
        value *= rate / (rate + s)
    return value


def exact_tail_probability(t):
    # The sum of independent Exp(1),...,Exp(d) variables is distributed as
    # the maximum of d independent Exp(1) variables. This form is stable.
    t = mp.mpf(t)
    return -mp.expm1(DIMENSION * mp.log1p(-mp.exp(-t)))


TRUE_TAIL = {int(t): float(exact_tail_probability(t)) for t in T_VALUES}


def learned_diagonal_zero_limit(model):
    zero = torch.zeros(1, DIMENSION, dtype=c_dtype, device=device)
    with torch.no_grad():
        _, jacobian = model.forward_with_input_jacobian(zero, need_jacobian=True)
    value = -torch.sum(jacobian[0, 0, :])
    return mp.mpc(float(value.real.detach().cpu()), float(value.imag.detach().cpu()))


def learned_tail_transform(model):
    zero_limit = learned_diagonal_zero_limit(model)

    @torch.no_grad()
    def phi(s):
        z = mp.mpc(s)
        theta = torch.full(
            (1, DIMENSION),
            complex(float(mp.re(z)), float(mp.im(z))),
            dtype=c_dtype,
            device=device,
        )
        value = torch.exp(model(theta)[0, 0])
        return mp.mpc(float(value.real.cpu()), float(value.imag.cpu()))

    def transform(s):
        s = mp.mpc(s)
        return zero_limit if s == 0 else (1 - phi(s)) / s

    return transform, zero_limit


def exact_tail_transform(s):
    s = mp.mpc(s)
    if s == 0:
        return mp.fsum(1 / rate for rate in RATES)
    return (1 - exact_product_form_transform(s)) / s


def parse_prediction(value):
    value = mp.mpc(value)
    real = float(mp.re(value))
    imaginary = float(mp.im(value))
    invalid = (
        not math.isfinite(real)
        or not math.isfinite(imaginary)
        or abs(imaginary) > 1.0e-8
        or real < 0.0
        or real > 1.0
    )
    return real, imaginary, bool(invalid)


def relative_error(prediction, truth):
    return abs(prediction - truth) / abs(truth)


def interval_mre(rows, end_t):
    selected = [row for row in rows if int(row["t"]) <= int(end_t)]
    return float(np.mean([row["relative_error"] for row in selected]))


def evaluate_exact_transform_floor():
    rows = []
    for degree in TALBOT_DEGREES:
        for t in T_VALUES:
            raw = mp.invertlaplace(
                exact_tail_transform, mp.mpf(t), method="talbot", degree=degree
            )
            prediction, imaginary, invalid = parse_prediction(raw)
            truth = TRUE_TAIL[t]
            rows.append({
                "method": "exact_transform_floor",
                "degree": degree,
                "t": t,
                "true_value": truth,
                "raw_prediction": prediction,
                "raw_prediction_imaginary": imaginary,
                "relative_error": relative_error(prediction, truth),
                "invalid_flag": invalid,
            })
    return rows


def load_checkpoint_payload(checkpoint_path):
    checkpoint_path = Path(checkpoint_path)
    actual_hash = sha256_file(checkpoint_path)
    try:
        return torch.load(checkpoint_path, map_location=device, weights_only=True)
    except Exception:
        # Legacy pickle loading is enabled only for the exact released file.
        if (
            HISTORICAL_CHECKPOINT_SHA256 is None
            or actual_hash != HISTORICAL_CHECKPOINT_SHA256
        ):
            raise RuntimeError(
                "Refusing legacy pickle loading for an unverified checkpoint."
            )
        return torch.load(checkpoint_path, map_location=device, weights_only=False)


def validate_historical_fixed_checkpoint(payload):
    if payload.get("checkpoint_schema") != HISTORICAL_CHECKPOINT_SCHEMA:
        raise RuntimeError("Unexpected historical checkpoint schema.")
    if payload.get("experiment_variant") != HISTORICAL_EXPERIMENT_VARIANT:
        raise RuntimeError("Historical experiment variant mismatch.")
    if int(payload.get("completed_step", -1)) != CHECKPOINT_STEP:
        raise RuntimeError(f"Expected fixed step {CHECKPOINT_STEP}.")
    if int(payload.get("last_completed_step", -1)) != CHECKPOINT_STEP:
        raise RuntimeError("Historical checkpoint step fields disagree.")
    config = payload.get("config", {})
    expected = {
        "seed": PAPER_SEED,
        "eval_seed": TRAIN_CONFIG["eval_seed"],
        "d": DIMENSION,
        "coord_embed_dim": MODEL_CONFIG["coord_embed_dim"],
        "boundary_embed_dim": MODEL_CONFIG["boundary_embed_dim"],
        "interior_hidden_width": MODEL_CONFIG["interior_hidden_width"],
        "boundary_hidden_width": MODEL_CONFIG["boundary_hidden_width"],
        "pair_chunk_size": MODEL_CONFIG["pair_chunk_size"],
        "start_step": 0,
        "steps": TRAIN_STEPS,
        "final_step": CHECKPOINT_STEP,
        "batch_bar": TRAIN_CONFIG["batch_bar"],
        "batch_bdary": TRAIN_CONFIG["batch_bdary"],
        "batch_mono": TRAIN_CONFIG["batch_mono"],
        "eval_batch_bar": TRAIN_CONFIG["eval_batch_bar"],
        "eval_batch_bdary": TRAIN_CONFIG["eval_batch_bdary"],
        "eval_batch_mono": TRAIN_CONFIG["eval_batch_mono"],
        "lr_stage1": TRAIN_CONFIG["lr_stage1"],
        "lr_stage2": TRAIN_CONFIG["lr_stage2"],
        "lr_transition_after_step": LR_SWITCH_AFTER_STEP,
        "grad_clip": TRAIN_CONFIG["grad_clip"],
    }
    for key, value in expected.items():
        if config.get(key) != value:
            raise RuntimeError(f"Historical configuration mismatch for {key!r}.")
    sampler = payload.get("training_sampler", {})
    if (
        sampler.get("name") != "original_general_mixed_BAR"
        or sampler.get("contains_special_inversion_nodes") is not False
    ):
        raise RuntimeError("Historical sampler metadata mismatch.")
    if payload.get("random_initialization") is not True:
        raise RuntimeError("Historical checkpoint is not from the frozen initialization.")
    if payload.get("source_checkpoint") is not None:
        raise RuntimeError("Historical checkpoint unexpectedly cites a source model.")
    if payload.get("optimizer_state_inherited") is not False:
        raise RuntimeError("Historical checkpoint inherited an external optimizer state.")


def load_fixed_model(checkpoint_path):
    checkpoint_path = Path(checkpoint_path)
    payload = load_checkpoint_payload(checkpoint_path)
    schema = payload.get("schema")
    if schema == "hcn_d50_fixed100k_v1":
        if int(payload.get("last_completed_step", -1)) != CHECKPOINT_STEP:
            raise RuntimeError(f"Expected fixed step {CHECKPOINT_STEP}.")
        if int(payload.get("seed", -1)) != PAPER_SEED:
            raise RuntimeError("Checkpoint seed mismatch.")
        if payload.get("model_config") != MODEL_CONFIG:
            raise RuntimeError("Checkpoint model configuration mismatch.")
        if payload.get("training_config") != TRAIN_CONFIG:
            raise RuntimeError("Checkpoint training configuration mismatch.")
        checkpoint_kind = "consolidated_public_training"
    elif payload.get("checkpoint_schema") == HISTORICAL_CHECKPOINT_SCHEMA:
        if HISTORICAL_CHECKPOINT_SHA256 is None:
            raise RuntimeError(
                "The archived checkpoint SHA-256 has not been frozen in this release."
            )
        if sha256_file(checkpoint_path) != HISTORICAL_CHECKPOINT_SHA256:
            raise RuntimeError("Historical checkpoint SHA-256 mismatch.")
        validate_historical_fixed_checkpoint(payload)
        checkpoint_kind = "audited_historical_fixed_endpoint"
    else:
        raise RuntimeError("Unexpected checkpoint schema.")
    model = build_model()
    model.load_state_dict(payload["model_state_dict"], strict=True)
    model.eval()
    return model, checkpoint_kind


def evaluate_checkpoint(checkpoint_path):
    checkpoint_path = Path(checkpoint_path)
    hash_before = sha256_file(checkpoint_path)
    model, checkpoint_kind = load_fixed_model(checkpoint_path)
    transform, zero_limit = learned_tail_transform(model)
    rows = []
    started = time.perf_counter()
    for degree in TALBOT_DEGREES:
        for t in T_VALUES:
            raw = mp.invertlaplace(
                transform, mp.mpf(t), method="talbot", degree=degree
            )
            prediction, imaginary, invalid = parse_prediction(raw)
            truth = TRUE_TAIL[t]
            rows.append({
                "benchmark": EXPERIMENT_SLUG,
                "seed": PAPER_SEED,
                "step": CHECKPOINT_STEP,
                "degree": degree,
                "t": t,
                "true_value": truth,
                "raw_prediction": prediction,
                "raw_prediction_imaginary": imaginary,
                "relative_error": relative_error(prediction, truth),
                "invalid_flag": invalid,
                "checkpoint_path": str(checkpoint_path),
                "checkpoint_sha256": hash_before,
            })
    elapsed = time.perf_counter() - started
    hash_after = sha256_file(checkpoint_path)
    if hash_before != hash_after:
        raise AssertionError("Evaluation modified the checkpoint.")
    summaries = []
    for degree in TALBOT_DEGREES:
        selected = [row for row in rows if row["degree"] == degree]
        summaries.append({
            "benchmark": EXPERIMENT_SLUG,
            "seed": PAPER_SEED,
            "step": CHECKPOINT_STEP,
            "degree": degree,
            "mre_1_5": interval_mre(selected, 5),
            "mre_1_8": interval_mre(selected, 8),
            "mre_1_10": interval_mre(selected, 10),
            "invalid_count_1_10": int(sum(row["invalid_flag"] for row in selected)),
            "evaluation_seconds": elapsed,
            "learned_zero_limit_real": float(mp.re(zero_limit)),
            "learned_zero_limit_imaginary": float(mp.im(zero_limit)),
            "checkpoint_sha256": hash_before,
            "checkpoint_kind": checkpoint_kind,
        })
    return rows, summaries


def save_evaluation_plots(run_dir, pointwise_rows):
    for degree in TALBOT_DEGREES:
        selected = [row for row in pointwise_rows if row["degree"] == degree]
        plotted = [
            row["raw_prediction"]
            if math.isfinite(row["raw_prediction"]) and row["raw_prediction"] > 0
            else np.nan
            for row in selected
        ]
        plt.figure(figsize=(9.0, 5.5))
        plt.semilogy(T_VALUES, [TRUE_TAIL[t] for t in T_VALUES], "k-o", label="exact tail")
        plt.semilogy(T_VALUES, plotted, "C0-x", label=f"HCN seed {PAPER_SEED}")
        plt.xlabel("t")
        plt.ylabel("tail probability (raw)")
        plt.grid(True, which="both", alpha=0.3)
        plt.legend()
        plt.tight_layout()
        plt.savefig(Path(run_dir) / f"tail_probability_degree{degree}.png", dpi=180)
        plt.close()

        plt.figure(figsize=(9.0, 5.5))
        plt.semilogy(
            T_VALUES, [row["relative_error"] for row in selected],
            "C0-o", label=f"HCN seed {PAPER_SEED}",
        )
        plt.xlabel("t")
        plt.ylabel("raw relative error")
        plt.grid(True, which="both", alpha=0.3)
        plt.legend()
        plt.tight_layout()
        plt.savefig(Path(run_dir) / f"relative_error_degree{degree}.png", dpi=180)
        plt.close()


def locate_fixed_endpoint(run_dir):
    run_dir = Path(run_dir).expanduser().resolve()
    candidates = [
        run_dir / f"checkpoint_step{CHECKPOINT_STEP}.pt",
        run_dir / f"cauchy_product_form_d50_general_pointmatched_step{CHECKPOINT_STEP}.pt",
        run_dir / f"d50_general_step{CHECKPOINT_STEP}.pt",
        run_dir / f"seed_{PAPER_SEED}" / f"checkpoint_step{CHECKPOINT_STEP}.pt",
    ]
    existing = [path for path in candidates if path.is_file()]
    if len(existing) != 1:
        raise FileNotFoundError(
            f"Expected exactly one fixed step-{CHECKPOINT_STEP:,} checkpoint; found: "
            + ", ".join(str(path) for path in existing)
        )
    return existing[0]


def new_evaluation_output_dir():
    OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
    output = OUTPUT_ROOT / f"{EXPERIMENT_SLUG}_evaluation_{utc_stamp()}"
    output.mkdir(parents=False, exist_ok=False)
    return output


def evaluate_run(run_dir, output_dir=None):
    source_run_dir = Path(run_dir).expanduser().resolve()
    checkpoint = locate_fixed_endpoint(source_run_dir)
    output_dir = (
        source_run_dir
        if output_dir is None
        else Path(output_dir).expanduser().resolve()
    )
    output_dir.mkdir(parents=True, exist_ok=True)
    pointwise_rows, summary_rows = evaluate_checkpoint(checkpoint)
    exact_rows = evaluate_exact_transform_floor()
    exact_summary = []
    for degree in TALBOT_DEGREES:
        selected = [row for row in exact_rows if row["degree"] == degree]
        exact_summary.append({
            "degree": degree,
            "mre_1_5": interval_mre(selected, 5),
            "mre_1_8": interval_mre(selected, 8),
            "mre_1_10": interval_mre(selected, 10),
            "invalid_count_1_10": int(sum(row["invalid_flag"] for row in selected)),
        })

    write_csv(output_dir / "checkpoint_pointwise.csv", pointwise_rows)
    write_csv(output_dir / "checkpoint_summary.csv", summary_rows)
    write_csv(output_dir / "exact_transform_talbot_floor.csv", exact_rows)
    write_csv(output_dir / "exact_transform_talbot_floor_summary.csv", exact_summary)
    save_evaluation_plots(output_dir, pointwise_rows)

    final_summary = {
        "primary_endpoint": f"fixed step {CHECKPOINT_STEP}",
        "paper_seed": PAPER_SEED,
        "primary_talbot_degree": EVALUATION_CONFIG["primary_degree"],
        "ground_truth_used_in_training": False,
        "ground_truth_used_for_checkpoint_selection": False,
        "raw_predictions": True,
        "prediction_clipping": False,
        "parameter_real_scalar_dof": PARAMETER_AUDIT["trainable_real_scalar_dof"],
        "total_sampled_constraints": PER_RUN_SAMPLE_BUDGET,
        "checkpoint_summary": summary_rows,
        "exact_transform_floor": exact_summary,
        "source_run_directory": str(source_run_dir),
        "evaluation_output_directory": str(output_dir),
    }
    write_json(output_dir / "evaluation_summary.json", final_summary)
    print(json.dumps(final_summary, indent=2))
    return final_summary


EVALUATION_SUMMARY = None
if RUN_MODE == "train":
    EVALUATION_SUMMARY = evaluate_run(FORMAL_RUN_DIR)
elif RUN_MODE == "evaluate":
    EVALUATION_SUMMARY = evaluate_run(
        EVALUATE_RUN_DIR,
        output_dir=new_evaluation_output_dir(),
    )
else:
    print("Smoke test complete. No formal results were written.")
